In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
### =======================================================================
### AbAgInteractionPredictor                                       
### Comprehensive antibody-antigen (Ab-Ag) interaction analysis framework.
### =======================================================================

# Basic imports

from gbmContacts import RiskLevel, InteractionType, PredictionResult, AbAgInteractionPredictor

import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Dict, List, Tuple, Union, Optional
from dataclasses import dataclass
from enum import Enum
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')

# ML and data processing.
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, roc_curve, auc
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# Plotting and visualization.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

# Scipy for smoothing.
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import UnivariateSpline

# LightGBM (optional but recommended).
try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False
    warnings.warn("LightGBM not installed. Install with: pip install lightgbm")

# PDF generation.
try:
    from reportlab.lib.pagesizes import letter, A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY
    from reportlab.lib import colors
    HAS_REPORTLAB = True
except ImportError:
    HAS_REPORTLAB = False
    warnings.warn("ReportLab not installed. PDF reports unavailable. Install with: pip install reportlab")

In [ ]:
# Data loading.

contact_libs = "C:/Users/user/Desktop/gbmContacts/data/"

affinity = pd.read_csv(contact_libs + "Affinity_data.txt", sep = "\t")
metadata = pd.read_csv(contact_libs + "Affinity_meta.txt", sep = "\t")
contact_class = pd.read_csv(contact_libs + "Contact_class.txt", sep = "\t")
contact_map = pd.read_csv(contact_libs + "Contact_map.txt", sep = "\t")
mutations = pd.read_csv(contact_libs + "Contact_mutations.txt", sep = "\t")
properties = pd.read_csv(contact_libs + "Aminoacid_properties.txt", sep = "\t")

In [ ]:
# AbAgInteractionPredictor instance.
abag = AbAgInteractionPredictor(
    affinity_data = affinity,
    affinity_metadata = metadata,
    contact_class = contact_class,
    contact_map = contact_map,
    contact_mutations = mutations,
    aa_properties = properties
)

In [ ]:
# Inspect contact_map_lookup library.
abag.contact_map_lookup

In [ ]:
# Loading input contacts.
contacts = abag.load_contacts_from_file(contact_libs + "unstable_complex_example.txt")
contacts

In [ ]:
# Interaction model training.
abag_fit = abag.fit()

In [ ]:
# Ab-Ag complex stability prediction.
abag_predict = abag.predict(contacts)

In [ ]:
# Plot report in PNG format.
abag_trend = abag.plot_interaction_trend(abag_predict, output_file = "C:/Users/user/Desktop/AbAg_prediction_report.png")

In [ ]:
# Generate PDF report
abag.generate_pdf_report(abag_predict, output_file = "C:/Users/user/Desktop/AbAg_prediction_report.pdf")